# MLflow: 5 تجارب
قبل ما تبدأ: `Runtime → Change runtime type → T4 GPU`.
في Drive جوه `mlops` لازم يكون: `src` (الجديد، فيه tracking.py) و`configs` و`scripts` و`requirements.txt` و`processed`.
الـnotebook بيكمّل من حيث وقف: لو الـsession فصل، شغّل الخلايا تاني من الأول وهيعدّي التجارب اللي خلصت.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/mlops'
import os, shutil
assert os.path.isdir(DRIVE), f"مش لاقي {DRIVE}"
WORK = '/content/project'
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK + '/data')
for name in ['src', 'configs', 'scripts']:
    shutil.copytree(f'{DRIVE}/{name}', f'{WORK}/{name}')
shutil.copy(f'{DRIVE}/requirements.txt', WORK)
shutil.copytree(f'{DRIVE}/processed', f'{WORK}/data/processed')
os.chdir(WORK)
if os.path.exists(f'{DRIVE}/mlflow.db'):          # استرجاع نتائج التجارب السابقة
    shutil.copy(f'{DRIVE}/mlflow.db', f'{WORK}/mlflow.db')
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
print("تمام")

In [ ]:
!pip install -q 'transformers>=4.46' accelerate scikit-learn pandas pyyaml mlflow

In [ ]:
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "مفيش GPU! غيّر الـRuntime لـT4 وشغّل من الأول"

## التجارب
`RUNS` تحدد أنهي تجارب تشتغل دلوقتي. تقدر تشغّل جزء وتكمّل بعدين.

In [ ]:
SAMPLES = 60000
RUNS = [1, 2, 3, 4, 5]
CONFIGS = {1: (2e-5, 8, 2), 2: (3e-5, 8, 2), 3: (5e-5, 8, 3), 4: (3e-5, 16, 2), 5: (3e-5, 16, 3)}  # lr, batch, epochs
TRACKING = f"sqlite:///{WORK}/mlflow.db"

import time
for i in RUNS:
    lr, bs, ep = CONFIGS[i]
    drive_run = f'{DRIVE}/runs/run_{i}'
    if os.path.exists(f'{drive_run}/metrics.json'):
        print(f"run {i}: خلصت قبل كده، هعدّيها")
        continue
    out = f'models/run_{i}'
    name = f'run{i}_lr{lr}_bs{bs}_ep{ep}'
    print(f"\n===== {name} =====")
    t = time.time()
    cmd = (f"python -m src.train --max-samples {SAMPLES} --learning-rate {lr} --batch-size {bs} --epochs {ep} "
           f"--output-dir {out} --model-version run{i} --run-name {name} --experiment arabic-sentiment --tracking-uri {TRACKING}")
    !{cmd}
    assert os.path.exists(f'{out}/metrics.json'), f"run {i} فشلت، شوف الـerror اللي فوق"
    shutil.copytree(out, drive_run, ignore=shutil.ignore_patterns('checkpoints'))
    shutil.copy(f'{WORK}/mlflow.db', f'{DRIVE}/mlflow.db')
    print(f"run {i} خلصت في {(time.time()-t)/60:.1f} دقيقة وانحفظت في Drive")

## النتائج واختيار الأفضل
الاختيار بـ **val_f1_macro** (مش الـtest) عشان الـtest يفضل نظيف للتقرير النهائي.

In [ ]:
import mlflow, json
mlflow.set_tracking_uri(TRACKING)
df = mlflow.search_runs(experiment_names=['arabic-sentiment'])
cols = {'tags.mlflow.runName': 'run', 'params.learning_rate': 'lr', 'params.batch_size': 'bs', 'params.epochs': 'epochs',
        'metrics.val_f1_macro': 'val_f1', 'metrics.test_f1_macro': 'test_f1', 'metrics.test_accuracy': 'test_acc',
        'metrics.f1_neutral': 'f1_neutral', 'metrics.train_seconds': 'seconds'}
table = df[list(cols)].rename(columns=cols).sort_values('val_f1', ascending=False).reset_index(drop=True)
display(table.round(4))

best = table.iloc[0]['run']
idx = int(best.split('_')[0].replace('run', ''))
dst = f'{DRIVE}/models/v2'
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(f'{DRIVE}/runs/run_{idx}', dst)
m = json.load(open(f'{dst}/metrics.json')); m['model_version'] = 'v2'
json.dump(m, open(f'{dst}/metrics.json', 'w'), indent=2, ensure_ascii=False)
shutil.copy(f'{WORK}/mlflow.db', f'{DRIVE}/mlflow.db')
print(f"\nالأفضل: {best}  ->  اتحفظ في Drive/mlops/models/v2")